In [0]:
%sql
CREATE OR REPLACE TABLE data_warehouse_factory.silver.silver_production_structure AS
WITH ranked_cells AS (
  SELECT 
    -- Unikalny klucz główny dla komórki produkcyjnej
    md5(concat_ws('||', upper(trim(line_code)), upper(trim(cell_code)))) AS cell_key,
    
    record_type,
    upper(trim(line_code)) AS line_code,
    trim(line_name) AS line_name,
    upper(trim(cell_code)) AS cell_code,
    trim(cell_name) AS cell_name,
    
    _source_file,
    _ingested_at AS _bronze_ingested_at,
    
    -- Numeracja w celu wyciągnięcia najświeższego rekordu per komórka
    ROW_NUMBER() OVER (
      PARTITION BY line_code, cell_code 
      ORDER BY _ingested_at DESC
    ) AS rnk
  FROM data_warehouse_factory.bronze.bronze_production_structure
  WHERE line_code IS NOT NULL 
    AND cell_code IS NOT NULL
)
SELECT 
  cell_key,
  record_type,
  line_code,
  line_name,
  cell_code,
  cell_name,
  _source_file,
  _bronze_ingested_at,
  current_timestamp() AS _silver_ingested_at
FROM ranked_cells
WHERE rnk = 1;